# Instructor solutions: Prinz MSM construction and validation

This self-contained notebook includes the worked lecture followed by a completed simulation-based homework solution. Run all cells in order. The solution uses the same reversible Metropolis-chain discretization; all time units are MC steps. Simulation and fitting require several minutes on a typical Colab CPU.

The final section reports measured results rather than assuming that more data automatically meets a target.


## 1. Equilibrium and the potential
For a canonical ensemble (paper Eq. 3),
$$\mu(x)=Z^{-1}e^{-\beta H(x)},\qquad \beta=(k_BT)^{-1}.$$
For the one-dimensional configurational example, replace the Hamiltonian by the potential:
$$V(x)=2x^8+0.8e^{-80x^2}+0.2e^{-80(x-0.5)^2}+0.5e^{-40(x+0.5)^2}.$$
We use $x\in[-1,1]$, $\beta=5$. These are the **Prinz-style lecture variant's** parameters.

On the reference grid, $w_j\propto e^{-\beta V(x_j)}$ are **probability masses** with $\sum_jw_j=1$. We reserve $\pi_s$ for the equilibrium probability of a basin, $\mu(x)$ for a continuous density, and $u=p/\pi$ for relative population. Position and energy are dimensionless.

In [ ]:
import sys
import numpy as np
import scipy
from scipy.linalg import eigh
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Python", sys.version.split()[0], "NumPy", np.__version__,
      "SciPy", scipy.__version__, "Matplotlib", matplotlib.__version__)

def potential(x):
    """Explicit lecture variant of the four-well Prinz potential."""
    return (2*x**8 + 0.8*np.exp(-80*x**2)
            + 0.2*np.exp(-80*(x-0.5)**2)
            + 0.5*np.exp(-40*(x+0.5)**2))

N_REFERENCE = 160
n = N_REFERENCE
beta = 5.0
proposal = 0.25  # Probability of proposing each neighboring move.
x = np.linspace(-1.0, 1.0, n)
energy = potential(x)

from scipy.integrate import quad
from IPython.display import display, HTML
from pathlib import Path
np.set_printoptions(precision=6, suppress=True)
COLORS = ["#8B0000", "#cf7c19", "#78952f", "#006080"]
BASINS = list("ABCD")
BASIN_EDGES = np.array([-1., -.5, 0., .5, 1.])

## 2. Fix the reference dynamics before building MSMs
Propose each neighboring move with probability $q=1/4$, accept with
$$P_{ij}=q\min(1,e^{-\beta(V_j-V_i)}),\qquad P_{ii}=1-\sum_{j\ne i}P_{ij}.$$
Proposals outside the domain are rejected. One attempt, including a rejection, is one MC step. Detailed balance (paper Eq. 4) is
$$\mu(x)p(x,y;\tau)=\mu(y)p(y,x;\tau),\qquad w_iP_{ij}=w_jP_{ji}.$$
The grid chain is connected and lazy: its equilibrium is unique and its spectrum is nonnegative. Reversibility alone does not imply a nonnegative spectrum.

**Homework invariant:** do not change this grid, potential, temperature or proposal to improve a model. Refine the *MSM partition of the same process*. Changing grid spacing changes the distance moved per MC step and therefore changes the kinetics.

In [ ]:
def metropolis_chain(energy, beta=5.0, proposal=0.25):
    """Return equilibrium masses and a lazy, nearest-neighbor transition matrix."""
    if beta < 0 or not 0 < proposal <= 0.25:
        raise ValueError("Use beta >= 0 and 0 < proposal <= 0.25")
    weights = np.exp(-beta * (energy - energy.min()))
    pi = weights / weights.sum()
    P = np.zeros((len(energy), len(energy)))
    for i in range(len(energy)):
        for j in (i-1, i+1):
            if 0 <= j < len(energy):
                # min(0, ...) avoids overflow for downhill moves.
                P[i, j] = proposal*np.exp(min(0., -beta*(energy[j]-energy[i])))
        P[i, i] = 1.0-P[i].sum()  # Rejections and unattempted moves.
    return pi, P


w, P = metropolis_chain(energy, beta, proposal)
np.testing.assert_allclose(P.sum(1), 1, atol=1e-14)
np.testing.assert_allclose(w @ P, w, atol=1e-14)
np.testing.assert_allclose(w[:, None]*P, w[None, :]*P.T, atol=1e-14)
assert P.min() >= 0 and np.diag(P).min() >= .5-1e-14
basin_labels = np.digitize(x, BASIN_EDGES[1:-1])
H4 = np.eye(4)[basin_labels]
pi4 = w @ H4
print("Equilibrium basin probabilities π:", pi4)
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True, layout="constrained")
ax[0].plot(x, energy, "k"); ax[0].set_ylabel("Potential V(x)")
ax[1].plot(x, w/(x[1]-x[0]), "k--"); ax[1].set(xlabel="Position x", ylabel="Equilibrium density (grid)")
for a in ax:
    for edge in BASIN_EDGES[1:-1]: a.axvline(edge, color=".8", lw=.8)
for pos, name in zip([-.75,-.25,.25,.75], BASINS): ax[0].text(pos,1.8,name,ha="center")
plt.show()

## 3. The propagator and transfer operator: same dynamics, different representations
The conditional transition **density** $p(x,y;\tau)$ is different from the time-dependent ensemble density $p_t(x)$.

**Propagator (Eq. 5):**
$$p_{t+\tau}(y)=[\mathcal{Q}(\tau)p_t](y)=\int_\Omega p(x,y;\tau)p_t(x)\,dx.$$
**Relative density (Eq. 7):**
$$u_t(x)=p_t(x)/\mu(x),\qquad p_t(x)=\mu(x)u_t(x).$$
**Transfer operator (Eq. 6):**
$$u_{t+\tau}(y)=[\mathcal{T}(\tau)u_t](y)=\frac{1}{\mu(y)}\int_\Omega p(x,y;\tau)\mu(x)u_t(x)\,dx.$$
Thus $\mathcal{T}=M_\mu^{-1}\mathcal{Q} M_\mu$: multiply relative density by equilibrium, propagate, then divide by equilibrium. Both paths give the same physical population.

For a **row-stochastic matrix** $P$, rows are starting states and columns are destination states. Row probability vectors evolve as $p_{t+\tau}=p_tP^\tau$. With **column vectors**, $Q_\tau=(P^\tau)^T$ and $T_\tau=D_w^{-1}Q_\tau D_w=P^\tau$ by detailed balance. The heatmap of $P$ shows conditional transitions; the column-propagator matrix is its transpose.

The equilibrium-weighted inner product is
$$\langle f,g\rangle_w=\sum_jw_jf_jg_j.$$
Here $f$ and $g$ are any real functions on the states. Detailed balance makes $T$ self-adjoint in this inner product. This gives real eigenvalues and orthogonal modes; $Q$ has the same spectrum and is self-adjoint with the reciprocal equilibrium weight.

In [ ]:
lag = 1000
P_lag = np.linalg.matrix_power(P, lag)
Q_lag = P_lag.T
T_lag = Q_lag * w[None, :] / w[:, None]
np.testing.assert_allclose(T_lag, P_lag, atol=1e-12)
p_demo = w*(1-.65*np.tanh(x/.1)); p_demo /= p_demo.sum()
u_demo = p_demo/w
np.testing.assert_allclose(Q_lag @ p_demo, w*(T_lag @ u_demo), atol=1e-12)
print("Propagating p and propagating u give the same population.")

## 4. Reference spectrum and equilibrium-conditioned projection
For the reversible one-MC-step matrix $P$, define $D_w=\operatorname{diag}(w_1,\ldots,w_n)$. We diagonalize the symmetric matrix
$$S=D_w^{1/2}PD_w^{-1/2}.$$
If $Sv_i=\ell_i v_i$, then the relative-density and probability-density eigenvectors are
$$\psi_i=D_w^{-1/2}v_i,\qquad \phi_i=D_w\psi_i.$$
In the fine-grid representation, the one-step transfer matrix is $T_1=P$ and the column-propagator matrix is $Q_1=P^T$. Therefore,
$$T_1\psi_i=\ell_i\psi_i,\qquad Q_1\phi_i=\ell_i\phi_i,\qquad \phi_i(j)=w_j\psi_i(j).$$
Here $\ell_i$ is an eigenvalue for **one MC step**. At integer lag $\tau$ MC steps,
$$T_\tau=P^\tau,\qquad Q_\tau=(P^\tau)^T,\qquad \lambda_i(\tau)=\ell_i^\tau.$$
The corresponding eigenvectors are unchanged by taking powers. The coarse-state eigenvalues below are separately computed from the coarse matrix; they need not equal these reference eigenvalues.

Let $H_{js}=1$ if fine-grid state $j$ belongs to MSM state $s$, and zero otherwise. The equilibrium probability of MSM state $s$ is
$$\pi_s=\sum_j w_jH_{js},\qquad D_\pi=\operatorname{diag}(\pi_1,\ldots,\pi_M).$$
The equilibrium-conditioned transition matrix at integer elapsed time $t$ MC steps is
$$T_H(t)=D_\pi^{-1}H^TD_wP^tH.$$
Its entry $[T_H(t)]_{ab}$ is the probability of ending in MSM state $b$, given a start in state $a$ drawn from equilibrium conditioned on that state.

This two-time conditional probability is exact for the reference chain. **It does not imply that the coarse-state process is Markovian.** Repeatedly applying $T_H(\tau)$ can introduce discretization error because a basin label loses information about position within the basin. Exact Markov coarse-graining is possible for special partitions; it is not guaranteed by reversibility.

In [ ]:
root = np.sqrt(w)
S = root[:, None]*P/root[None, :]
np.testing.assert_allclose(S, S.T, atol=1e-14)
ell, Vref = eigh(S)
ell, Vref = ell[::-1], Vref[:, ::-1]
psi_ref = Vref/root[:, None]
# Orient the leading modes to match the lecture. Signs are otherwise arbitrary.
for j, anchor in enumerate([0, n-1, 0, n-1]):
    if psi_ref[anchor, j] < 0:
        Vref[:, j] *= -1; psi_ref[:, j] *= -1
phi_ref = w[:, None]*psi_ref
np.testing.assert_allclose(P @ psi_ref, psi_ref*ell, atol=1e-10)
np.testing.assert_allclose(P.T @ phi_ref, phi_ref*ell, atol=1e-12)
np.testing.assert_allclose(psi_ref.T @ (w[:, None]*psi_ref), np.eye(n), atol=1e-12)

def projected_matrix(H, lag):
    """Exact reference conditional probabilities, not a trajectory estimate."""
    if int(lag) != lag or lag < 0: raise ValueError("lag must be a nonnegative integer")
    masses = w @ H
    if np.any(masses <= 0): raise ValueError("Partition has empty states")
    B = Vref.T @ (root[:, None]*H)
    # Spectral powers make repeated evaluations cheap without changing the result.
    joint = B.T @ ((ell**int(lag))[:, None]*B)
    return joint/masses[:, None]

def T4(lag): return projected_matrix(H4, lag)
for lag in [0, 100, 1000]:
    direct = (H4.T @ (w[:, None]*np.linalg.matrix_power(P, lag)) @ H4)/pi4[:, None]
    np.testing.assert_allclose(T4(lag), direct, atol=2e-12)
    np.testing.assert_allclose(T4(lag).sum(1), 1, atol=2e-12)
    np.testing.assert_allclose(pi4[:,None]*T4(lag), pi4[None,:]*T4(lag).T, atol=2e-12)
print("T₄(1000), rows and columns A, B, C, D:\n", T4(1000))
fig, ax = plt.subplots(1,2,figsize=(11,4),layout="constrained")
im=ax[0].imshow(P_lag,origin="upper",aspect="auto");fig.colorbar(im,ax=ax[0])
ax[0].set(xlabel="Destination grid state",ylabel="Starting grid state",title="160 × 160: 1,000 MC steps")
im=ax[1].imshow(T4(1000),vmin=0,vmax=1);fig.colorbar(im,ax=ax[1])
ax[1].set(xticks=range(4),xticklabels=BASINS,yticks=range(4),yticklabels=BASINS,xlabel="Destination basin",ylabel="Starting basin",title="4 × 4: 1,000 MC steps")
for i in range(4):
    for j in range(4): ax[1].text(j,i,f"{T4(1000)[i,j]:.4f}",ha="center",va="center",color="white" if T4(1000)[i,j]<.5 else "black")
plt.show()

## 5. Repeated propagation and the CK test
For the full Markov process (Eqs. 8–9),
$$\mathcal{Q}(k\tau)=\mathcal{Q}(\tau)^k,\qquad\mathcal{T}(k\tau)=\mathcal{T}(\tau)^k.$$
For the four-state approximation we **test**, rather than assume,
$$[T_4(\tau)^k]_{ab}\approx[T_4(k\tau)]_{ab}.$$
Blue lines: repeated four-state prediction. Black dots: direct longer-time evaluation of the reference chain, then projection. **Do not construct the black dots by raising the same four-state matrix to a power**; that would be a tautology.

The diagonal is return probability, allowing departure and return. It is not a survival probability. Agreement at $k=1$ is guaranteed here because both use the same exact lag-time matrix. Differences at later times are discretization error, not sampling noise. The 100-step row covers 1,000 total steps; the 1,000-step row covers 10,000. Compare the elapsed MC steps, $k\tau$, when interpreting the two rows; these are not calibrated physical times.

In [ ]:
K = np.arange(11)
def ck_arrays(lag):
    direct = np.array([T4(int(k*lag)) for k in K])
    predicted = np.array([np.linalg.matrix_power(T4(lag), int(k)) for k in K])
    return direct, predicted

fig, ax = plt.subplots(2,4,figsize=(12,6),sharex=True,sharey=True,layout="constrained")
ck_max = {}
for row, lag in enumerate([100,1000]):
    direct,predicted=ck_arrays(lag)
    ck_max[lag]=np.max(abs(direct-predicted))
    for s in range(4):
        ax[row,s].plot(K,direct[:,s,s],"ko",ms=3,label="Direct reference")
        ax[row,s].plot(K,predicted[:,s,s],color="#0072b2",label="Four-state prediction")
        ax[row,s].set(title=f"Basin {BASINS[s]}",ylim=(0,1.03))
        if s==0: ax[row,s].set_ylabel(f"τ = {lag:,} MC steps\nReturn probability")
        if row==1: ax[row,s].set_xlabel("Lag steps k")
ax[0,0].legend(fontsize=8);plt.show()
print("Maximum absolute error across all 16 entries and k=0…10:", ck_max)
np.testing.assert_allclose([ck_max[100],ck_max[1000]],[.1509348438,.0239082],atol=2e-6)

## 6. CK tests for all 16 transitions
Rows are starting basins; columns are destination basins. Every panel uses the same probability scale. Small absolute errors on rare transitions can still be large *relative* errors.

At short lag, a basin label forgets where the particle is inside the well. Repeated propagation implicitly restores the equilibrium conditional distribution inside each basin. Real trajectories retain information about their entry position. More resolved states can preserve more of this information.

In [ ]:
def plot_ck_grid(direct, predicted, lag, title="Four-state CK test"):
    fig, axes=plt.subplots(4,4,figsize=(11,8),sharex=True,sharey=True,layout="constrained")
    for i in range(4):
        for j in range(4):
            ax=axes[i,j]
            ax.plot(K,direct[:,i,j],"ko",ms=3,label="Direct reference / validation")
            ax.plot(K,predicted[:,i,j],color="#0072b2",label="MSM prediction")
            ax.set(title=f"{BASINS[i]} → {BASINS[j]}",ylim=(0,1.03))
            if j==0: ax.set_ylabel("Probability")
            if i==3: ax.set_xlabel("Lag steps k")
    axes[0,0].legend(fontsize=7)
    fig.suptitle(f"{title}; τ = {lag:,} MC steps",fontsize=15)
    plt.show()
for lag in [100,1000]: plot_ck_grid(*ck_arrays(lag),lag)

## 7. The actual four-state eigenvectors
At the lecture lag $\tau=1000$,
$$T_4\psi_i=\lambda_i\psi_i,\quad\phi_i=\pi\psi_i,\quad T_4^T\phi_i=\lambda_i\phi_i.$$
We normalize $\sum_s\pi_s\psi_i(s)\psi_j(s)=\delta_{ij}$ and orient the signs to match the slides. The stationary mode is $\psi_1=1$, $\phi_1=\pi$.

A four-state eigenvector has four values. Its step-function plot is not the smooth reference eigenfunction. The two resolutions have different numerical eigenvalues. Positive and negative lobes identify opposite sides of an imbalance; the coefficient's sign determines which side is overpopulated.

In [ ]:
def reversible_modes(T, equilibrium):
    """Return descending eigenvalues and equilibrium-normalized right modes."""
    r=np.sqrt(equilibrium)
    sym=r[:,None]*T/r[None,:]
    np.testing.assert_allclose(sym,sym.T,atol=2e-11)
    values,vectors=eigh(sym)
    values=values[::-1]; vectors=vectors[:,::-1]/r[:,None]
    for j in range(vectors.shape[1]):
        anchor=np.argmax(abs(vectors[:,j]))
        if vectors[anchor,j]<0: vectors[:,j]*=-1
    return values,vectors

T_four=T4(1000)
lambda4,psi4=reversible_modes(T_four,pi4)
for j,anchor in enumerate([0,3,0,3]):
    if psi4[anchor,j]<0: psi4[:,j]*=-1
phi4=pi4[:,None]*psi4
print("Eigenvalues:",lambda4)
print("Right eigenvectors (rows A–D, columns modes 1–4):\n",psi4)
np.testing.assert_allclose(lambda4,[1,.96423042,.70241329,.43963485],atol=1e-8)
np.testing.assert_allclose(T_four@psi4,psi4*lambda4,atol=1e-12)
np.testing.assert_allclose(T_four.T@phi4,phi4*lambda4,atol=1e-12)
np.testing.assert_allclose(psi4.T@(pi4[:,None]*psi4),np.eye(4),atol=1e-12)
np.testing.assert_allclose(psi4[:,0],1,atol=1e-12)
fig,ax=plt.subplots(4,2,figsize=(10,8),layout="constrained")
for i in range(4):
    ax[i,0].plot(x,psi_ref[:,i],color=COLORS[i]);ax[i,0].set_ylabel(f"ψ{i+1}")
    ax[i,1].stairs(psi4[:,i],np.arange(5)-.5,color=COLORS[i],lw=2)
    ax[i,1].set_xticks(range(4),BASINS)
    for aa in ax[i]: aa.axhline(0,color=".7",lw=.6)
ax[0,0].set_title("Reference grid eigenfunctions")
ax[0,1].set_title("Four-state eigenvectors at τ = 1,000")
ax[-1,0].set_xlabel("Position x");ax[-1,1].set_xlabel("Basin");plt.show()

## 8. Eigenvalues and implied timescales
For positive nonstationary eigenvalues,
$$t_i(\tau)=-\frac{\tau}{\ln\lambda_i(\tau)}.$$
The stationary mode has infinite lifetime. A negative eigenvalue represents sign alternation, so this real exponential formula does not apply directly; do not silently take its absolute value. Our exact lazy reference chain avoids this complication.

The reference spectrum obeys lag invariance exactly. The coarse model's implied timescales can depend on lag. A plateau and a CK test are useful diagnostics, not proof of correctness. A spectral gap after four modes must be inspected, not inferred solely from the number of wells.

In [ ]:
lags=np.array([10,25,50,100,200,500,1000,2000,5000])
coarse_times=[]
for lag in lags:
    val,_=reversible_modes(T4(int(lag)),pi4)
    assert np.all((val[1:]>0)&(val[1:]<1))
    coarse_times.append(-lag/np.log(val[1:]))
coarse_times=np.array(coarse_times)
print("Four-state timescales at τ=1000 (MC steps):",-1000/np.log(lambda4[1:]))
fig,ax=plt.subplots(1,2,figsize=(11,4),layout="constrained")
ax[0].plot(np.arange(1,13),ell[:12]**1000,"ko-")
ax[0].set(xlabel="Mode index",ylabel="Eigenvalue",title="Reference spectrum at 1,000 MC steps")
for i in range(3):
    ax[1].plot(lags,coarse_times[:,i],"o-",color=COLORS[i+1],label=f"Mode {i+2}")
    ax[1].axhline(-1/np.log(ell[i+1]),color=COLORS[i+1],ls="--")
ax[1].set(xlabel="Lag (MC steps)",ylabel="Implied time (MC steps)",title="Four-state estimates; dashed = reference",xscale="log")
ax[1].legend();plt.show()

## 9. Reproduce the lecture's initial distribution
The smooth continuous initial density is deliberately biased to the left:
$$\mu_c(x)=\frac{e^{-5V(x)}}{\int_{-1}^{1}e^{-5V(z)}dz},\qquad
p_0(x)=\frac{\mu_c(x)[1-0.65\tanh(x/0.1)]}{\int_{-1}^{1}\mu_c(z)[1-0.65\tanh(z/0.1)]dz}.$$
Compute basin populations by **integration**, not by reading peak heights:
$$p_{0,s}=\int_s p_0(x)dx,\qquad u_{0,s}=p_{0,s}/\pi_s.$$
To reproduce the lecture's numbers exactly, integrate this smooth density and use the four-state model's grid-derived equilibrium probabilities. Continuous quadrature and finite-grid equilibrium differ slightly; we name them separately rather than treating them as identical.

In [ ]:
Z=quad(lambda z: np.exp(-beta*potential(z)),-1,1,epsabs=1e-12)[0]
def mu_cont(z): return np.exp(-beta*potential(z))/Z
Z_bias=quad(lambda z: mu_cont(z)*(1-.65*np.tanh(z/.1)),-1,1,epsabs=1e-12)[0]
def p0_cont(z): return mu_cont(z)*(1-.65*np.tanh(z/.1))/Z_bias
p0_four=np.array([quad(p0_cont,lo,hi,epsabs=1e-12)[0] for lo,hi in zip(BASIN_EDGES[:-1],BASIN_EDGES[1:])])
u0_four=p0_four/pi4
np.testing.assert_allclose(p0_four.sum(),1,atol=1e-12)
np.testing.assert_allclose(p0_four,[.2114466364,.5156871225,.1827922343,.0900740067],atol=1e-9)
print("Initial basin probabilities:",p0_four)
print("Relative populations:",u0_four)
xx=np.linspace(-1,1,2001)
fig,ax=plt.subplots(3,1,figsize=(8,7),sharex=True,layout="constrained")
ax[0].plot(xx,potential(xx),"k");ax[0].set_ylabel("Potential")
ax[1].plot(xx,mu_cont(xx),"k--",label="Equilibrium μ(x)")
ax[1].plot(xx,p0_cont(xx),color="#0072b2",label="Initial p₀(x)");ax[1].set_ylabel("Density");ax[1].legend()
ax[2].plot(xx,p0_cont(xx)/mu_cont(xx),color="#0072b2");ax[2].axhline(1,color="k",ls="--")
ax[2].set(xlabel="Position x",ylabel="Relative density u₀",ylim=(0,2.3))
for aa in ax:
    for edge in BASIN_EDGES[1:-1]: aa.axvline(edge,color=".8",lw=.7)
plt.show()

## 10. Calculate each amplitude, then reconstruct the imbalance
The continuous weighted inner product simplifies because $u_0\mu=p_0$:
$$a_i=\langle u_0,\psi_i\rangle_\mu=\int u_0\psi_i\mu\,dx=\int p_0\psi_i\,dx.$$
For our four-state modes,
$$a_i=\sum_s\pi_su_{0,s}\psi_i(s)=\sum_sp_{0,s}\psi_i(s).$$
**In words:** average the eigenvector's value over the starting population. These coefficients assume weighted-orthonormal modes. A mode's sign is arbitrary, but its coefficient changes sign with it, leaving $a_i\psi_i$ unchanged.

The lecture's mode 2 calculation is
$$a_2\approx(0.2114)(-1.4689)+(0.5157)(-1.2250)+(0.1828)(0.7438)+(0.0901)(0.8158)=-0.7329.$$
With all four modes there is no truncation remainder *within this four-state model*:
$$u_0(s)-1=\sum_{i=2}^{4}a_i\psi_i(s),\qquad p_{0,s}=\pi_su_0(s).$$
Negative $a_2$ multiplying negative mode values in A/B gives excess population on the left. Individual signed modes are not probability distributions.

In [ ]:
amplitudes=p0_four@psi4
products=p0_four[:,None]*psi4
print("Basin     p₀         π          u₀         ψ₂         p₀ ψ₂")
for s,name in enumerate(BASINS):
    print(f"{name:>3}   {p0_four[s]:9.6f} {pi4[s]:9.6f} {u0_four[s]:9.6f} {psi4[s,1]:10.6f} {products[s,1]:10.6f}")
print("Amplitudes (including stationary a₁=1):",amplitudes)
np.testing.assert_allclose(amplitudes,[1,-.7328719655,-.0368925123,.0140410981],atol=1e-9)
np.testing.assert_allclose(psi4@amplitudes,u0_four,atol=1e-12)
np.testing.assert_allclose(pi4*(psi4@amplitudes),p0_four,atol=1e-12)
fig,ax=plt.subplots(1,3,figsize=(12,3.7),layout="constrained")
ax[0].bar(BASINS,products[:,1],color=COLORS[1]);ax[0].set(title="Multiply population by mode value",ylabel="Contribution to a₂")
ax[1].bar(["Mode 2","Mode 3","Mode 4"],amplitudes[1:],color=COLORS[1:]);ax[1].set(title="Sum for each mode",ylabel="Amplitude")
for i in range(1,4): ax[2].plot(BASINS,amplitudes[i]*psi4[:,i],"o-",color=COLORS[i],label=f"a{i+1} ψ{i+1}")
ax[2].plot(BASINS,u0_four-1,"ko",mfc="none",ms=9,label="Initial imbalance")
ax[2].plot(BASINS,psi4[:,1:]@amplitudes[1:],"k--",label="Sum")
ax[2].set(title="Reconstruct u₀ − 1",ylabel="Relative population imbalance");ax[2].legend(fontsize=8)
for aa in ax: aa.axhline(0,color=".6",lw=.8)
plt.show()

## 11. Propagate the modes and rebuild the population
$$a_i(k)=a_i(0)\lambda_i^k,\quad u_k(s)=1+\sum_{i=2}^4a_i(k)\psi_i(s),\quad p_k(s)=\pi_su_k(s).$$
The equilibrium contribution stays fixed. The *imbalance* decays, not the total probability. These are predictions of repeated application of $T_4(1000)$, checked independently by matrix multiplication. They need not equal projection of the full process at $1000k$; that is the CK question.

The continuous density shown in the optional movie is a lift of basin populations using equilibrium shapes inside each basin. It preserves the initial basin probabilities but not every detail of the smooth initial curve. It is an instantaneous ensemble prediction, not the cumulative histogram of one MC trajectory.

In [ ]:
def population_at(k):
    if int(k)!=k or k<0: raise ValueError("Use a nonnegative integer lag count")
    coeff=amplitudes*(lambda4**int(k))
    relative=psi4@coeff
    return coeff,relative,pi4*relative

snapshots=[0,2,10,40,150]
fig,ax=plt.subplots(4,len(snapshots),figsize=(14,8),sharex=True,layout="constrained")
for col,k in enumerate(snapshots):
    coeff,relative,pop=population_at(k)
    np.testing.assert_allclose(pop,p0_four@np.linalg.matrix_power(T_four,k),atol=2e-12)
    np.testing.assert_allclose(pop.sum(),1,atol=2e-12);assert pop.min()>0
    for row in range(3):
        aa=ax[row,col];aa.bar(range(4),coeff[row+1]*psi4[:,row+1],color=COLORS[row+1])
        aa.axhline(0,color=".7",lw=.7)
        aa.set_ylim(*[(-.7,1.3),(-.11,.08),(-.02,.035)][row])
        aa.text(.98,.95,f"a{row+2}={coeff[row+1]:.3g}",transform=aa.transAxes,ha="right",va="top",fontsize=9)
        if col==0: aa.set_ylabel(f"Mode {row+2}")
        else: aa.tick_params(labelleft=False)
    ax[0,col].set_title(f"k = {k}\n{k*1000:,} MC steps")
    ax[3,col].bar(range(4),pop,color="#0072b2");ax[3,col].plot(range(4),pi4,"k_",ms=17,mew=2)
    ax[3,col].set(ylim=(0,.6),xticks=range(4),xticklabels=BASINS,xlabel="Basin")
    if col==0: ax[3,col].set_ylabel("Probability")
fig.suptitle("Mode contributions (row-specific scales); population approaches black equilibrium marks")
plt.show()

In [ ]:
# Optional browser-native movie. Set True and rerun this cell in Colab.
# It advances one lag at a time; there is no fitted trajectory or interpolation.
RUN_MOVIE = False
if RUN_MOVIE:
    from matplotlib.animation import FuncAnimation
    plt.rcParams["animation.embed_limit"]=50
    fig,ax=plt.subplots(1,3,figsize=(12,3.8),layout="constrained")
    density_line,=ax[0].plot(x,w/(x[1]-x[0]),color="#0072b2")
    ax[0].plot(x,w/(x[1]-x[0]),"k--");ax[0].set(xlim=(-1,1),ylim=(0,2.7),xlabel="x",ylabel="Lifted density")
    relative_line,=ax[1].step(range(4),np.ones(4),where="mid",color="#0072b2")
    ax[1].axhline(1,color="k",ls="--");ax[1].set(xticks=range(4),xticklabels=BASINS,ylim=(0,2.3),ylabel="Relative population")
    bars=ax[2].bar(range(4),p0_four,color="#0072b2");ax[2].plot(range(4),pi4,"k_",ms=18)
    ax[2].set(xticks=range(4),xticklabels=BASINS,ylim=(0,.6),ylabel="Basin probability")
    title=fig.suptitle("")
    def update_movie(k):
        coeff,relative,pop=population_at(k)
        density_line.set_ydata(w/(x[1]-x[0])*(H4@relative))
        relative_line.set_ydata(relative)
        for bar,height in zip(bars,pop): bar.set_height(height)
        title.set_text(f"k={k}, time={1000*k:,} MC steps; a₂={coeff[1]:.3g}, a₃={coeff[2]:.3g}, a₄={coeff[3]:.3g}")
    movie=FuncAnimation(fig,update_movie,frames=range(151),interval=200)
    html=movie.to_jshtml();plt.close(fig);display(HTML(html))

## 12. Discretization: compare models on the same four observables
A finer MSM has $M$ states but should still predict the **same A–D basin transitions**. Otherwise, changing the dimension changes the meaning of the error metric.

For a refinement $H_M$ that never crosses an A–D boundary, let $G$ map its states to basins and
$$R_{as}=\Pr(\text{fine state }s\mid\text{basin }a,\text{equilibrium}).$$
The finer model's prediction is
$$\widehat B(k\tau)=R\,[T_M(\tau)]^kG.$$
It must retain the $M$ states during propagation. Collapsing to four states first and then taking powers throws the resolution away.

Below is a common exact-reference scoring function. It reports absolute probability error across all 16 basin transitions. More states often help, but state placement matters too. At $M=160$ we recover the reference chain's exact Markov description, which is a useful control rather than the target homework solution.

In [ ]:
def uniform_partition(M):
    """Partition fixed grid states into contiguous, equally sized MSM states."""
    if M<4 or M>n or n%M or M%4:
        raise ValueError("Choose M dividing 160 and divisible by 4")
    return np.eye(M)[np.arange(n)*M//n]

def basin_maps(H, equilibrium=w):
    """Equilibrium basin-to-MSM initialization R, and MSM-to-basin map G."""
    # Each MSM state must belong to exactly one of the original four basins.
    overlap=H.T@H4
    if not np.all((overlap>0).sum(1)==1): raise ValueError("MSM states must refine A–D")
    G=(overlap>0).astype(float)
    basin_mass=equilibrium@H4
    R=(H4.T@(equilibrium[:,None]*H))/basin_mass[:,None]
    return R,G

def exact_partition_score(M, lag=100):
    H=uniform_partition(M);R,G=basin_maps(H)
    Tm=projected_matrix(H,lag)
    prediction=np.array([R@np.linalg.matrix_power(Tm,int(k))@G for k in K])
    reference=np.array([T4(int(k*lag)) for k in K])
    return float(np.max(abs(prediction-reference))),prediction

baseline_error,_=exact_partition_score(4)
print(f"Four-state baseline at lag 100: {baseline_error:.6f}")
# Full-resolution consistency check, not a refinement sweep / homework solution.
full_error,_=exact_partition_score(160)
assert full_error<1e-10
print(f"160-state reference control: {full_error:.2e}")

## 13. Simulation-based MSM construction: supplied building blocks
Now introduce **sampling error** by estimating transitions from MC trajectories. A longer trajectory increases transition counts. It cannot eliminate the systematic four-state projection error. A finer partition reduces lost positional information, but usually needs more data.

The sampler below uses exactly the reference chain $P$, including rejections. Save every MC step. Different trajectories must be counted separately: never count a transition across a concatenation boundary. Seeds are explicit.

For counts $C_{ij}(\tau)$, we use the transparent reversible estimator
$$F=(C+C^T)/2,\quad\widehat\pi_i=\frac{\sum_jF_{ij}}{\sum_{ij}F_{ij}},\quad\widehat T_{ij}=F_{ij}/\sum_jF_{ij}.$$
This is **symmetrized-count estimation, not the reversible maximum-likelihood estimator**. It satisfies detailed balance by construction and is consistent for adequately sampled equilibrium trajectories. No pseudocounts are silently added. Unvisited or disconnected states are flagged; run more data or reconsider the partition rather than quietly deleting states or inventing transitions.

In [ ]:
from scipy.sparse.csgraph import connected_components

def simulate_mc(n_steps, seed, start=None):
    """One fixed-reference trajectory; includes the state at time zero."""
    rng=np.random.default_rng(seed)
    state=int(rng.choice(n,p=w)) if start is None else int(start)
    path=np.empty(n_steps+1,dtype=np.int16);path[0]=state
    draws=rng.random(n_steps)
    for t,rnd in enumerate(draws,1):
        left=P[state,state-1] if state>0 else 0.
        right=P[state,state+1] if state<n-1 else 0.
        if rnd<left: state-=1
        elif rnd<left+right: state+=1
        path[t]=state
    return path

def transition_counts(paths, H, lag):
    """Sliding-window counts; paths remain independent at their boundaries."""
    if int(lag)!=lag or lag<1: raise ValueError("lag must be a positive integer")
    lag=int(lag);M=H.shape[1];assignment=H.argmax(1)
    counts=np.zeros((M,M),dtype=np.int64)
    for path in paths:
        states=assignment[np.asarray(path)]
        if len(states)<=lag: continue
        counts+=np.bincount(states[:-lag]*M+states[lag:],minlength=M*M).reshape(M,M)
    return counts

def fit_reversible_counts(counts):
    flux=(counts+counts.T)/2
    totals=flux.sum(1)
    if np.any(totals==0): raise ValueError("Unvisited states: collect more transitions")
    if connected_components(flux>0,directed=False,return_labels=False)!=1:
        raise ValueError("Disconnected count graph: collect inter-state transitions")
    fitted=flux/totals[:,None];stationary=totals/totals.sum()
    np.testing.assert_allclose(stationary[:,None]*fitted,stationary[None,:]*fitted.T,atol=1e-14)
    return fitted,stationary

def empirical_basin_transitions(paths, lag):
    counts=transition_counts(paths,H4,lag)
    if np.any(counts.sum(1)==0): raise ValueError("No validation origins in some basin")
    return counts/counts.sum(1)[:,None]

# Small deterministic checks catch lag and trajectory-boundary mistakes.
identity=np.eye(n)
c_test=transition_counts([np.array([0,1,2]),np.array([3,4,5])],identity,1)
assert c_test.sum()==4 and c_test[2,3]==0
assert transition_counts([np.array([0,1,2,3])],identity,2)[0,2]==1
path_check=simulate_mc(1000,seed=42)
assert np.all(abs(np.diff(path_check))<=1)
assert np.array_equal(path_check,simulate_mc(1000,seed=42))

### A short trajectory-based example (four-state baseline)
We start independent trajectories from the known equilibrium distribution, so burn-in is unnecessary for this controlled example. This does not make frames independent: MC samples remain correlated. For arbitrary starts, assess equilibration and discard an appropriate burn-in.

The training example is intentionally modest. The homework asks you to extend it. The four-state fit below is not expected to meet the refined-model target. We print count diagnostics and leading timescales rather than accepting a fit solely because it returns eigenvalues.

In [ ]:
training_demo=[simulate_mc(100_000,seed=seed) for seed in [10,11,12,13]]
counts_demo=transition_counts(training_demo,H4,100)
T_demo,pi_demo=fit_reversible_counts(counts_demo)
values_demo,modes_demo=reversible_modes(T_demo,pi_demo)
print("Training transitions counted:",counts_demo.sum())
print("Counts by starting basin:",counts_demo.sum(1))
print("Estimated equilibrium:",pi_demo)
print("Estimated eigenvalues:",values_demo)
print("Timescales for eigenvalues in (0,1):")
for i,value in enumerate(values_demo[1:],2):
    print(i, -100/np.log(value) if 0<value<1 else "not a positive relaxation eigenvalue")
print("This demonstration is not a converged homework solution.")

In [ ]:
# Baseline plots from trajectory counts; reuse this pattern for finer partitions.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7), constrained_layout=True)
for i in range(4):
    axes[0].plot(np.arange(4), modes_demo[:, i], marker="o", label=f"Mode {i+1}")
axes[0].set(xticks=np.arange(4), xticklabels=list("ABCD"), xlabel="Basin", ylabel="Mode value", title="Four-state modes estimated from MC")
axes[0].legend()
lag_grid = np.array([25, 50, 100, 200, 500, 1000])
fitted_times = []
for lag in lag_grid:
    fitted, stationary = fit_reversible_counts(transition_counts(training_demo, H4, int(lag)))
    values, _ = reversible_modes(fitted, stationary)
    fitted_times.append([-lag/np.log(v) if 0 < v < 1 else np.nan for v in values[1:4]])
for i in range(3):
    axes[1].plot(lag_grid, np.array(fitted_times)[:, i], "o-", label=f"Mode {i+2}")
axes[1].set(xlabel="Lag (MC steps)", ylabel="Implied timescale (MC steps)", title="Refit from counts at each lag")
axes[1].legend()
plt.show()


### Homework helper: evaluate a fitted MSM without collapsing it too early
For the main benchmark we supply the exact equilibrium-conditional initializer $R$ so that all models are tested with the same starting ensembles. Only the transition matrix is fitted. This is a controlled diagnostic, not an assertion that equilibrium is known in a general molecular problem.

As an extension, estimate $R$ from training occupancies within each basin and report how uncertainty in initialization changes predictions. The validator below never uses the validation data for fitting.

In [ ]:
def evaluate_fitted_msm(T_fitted, H, validation_paths, lag=100):
    R,G=basin_maps(H)  # Known reference initialization, explicitly supplied.
    predicted=np.array([R@np.linalg.matrix_power(T_fitted,int(k))@G for k in K])
    exact=np.array([T4(int(k*lag)) for k in K])
    observed=np.array([np.eye(4) if k==0 else empirical_basin_transitions(validation_paths,int(k*lag)) for k in K])
    return {"predicted":predicted,"validation":observed,"exact":exact,
            "max_error_vs_exact":float(np.max(abs(predicted[1:]-exact[1:]))),
            "max_error_vs_validation":float(np.max(abs(predicted[1:]-observed[1:])))}

# Student workflow (uncomment and choose your own partition and budgets):
# H = uniform_partition(M)
# training = [simulate_mc(steps_per_path, seed=s) for s in training_seeds]
# validation = [simulate_mc(steps_per_path, seed=s) for s in validation_seeds]
# counts = transition_counts(training, H, lag=100)
# fitted, fitted_pi = fit_reversible_counts(counts)
# result = evaluate_fitted_msm(fitted, H, validation, lag=100)
# plot_ck_grid(result["validation"], result["predicted"], 100,
#              title="Held-out trajectory validation")
# values, modes = reversible_modes(fitted, fitted_pi)
# Plot modes[:, :4] on the partition and repeat the fit at other lags.

## Numerical validation and limits
The executed lesson checks stochasticity, stationarity, detailed balance, the relationship between $Q$ and $T$, reference and coarse eigenpair residuals, weighted orthogonality, spectral versus matrix propagation, lecture values for eigenvalues and amplitudes, probability conservation, the 160-state CK control, and transition-count boundary handling.

The computed lecture figures use exact reference probabilities. The simulation section introduces a simple reversible count estimator. It does not implement a reversible maximum-likelihood estimator or provide a finished homework solution. The homework asks for empirical uncertainty rather than supplying misleading independent-transition error bars.

The original paper's exact diffusion parameters, theoretical error-bound figures and molecular systems are outside this numerical reproduction. The notebook follows the lecture's theory and generated Metropolis examples; it does not claim to reproduce every imported paper figure.

Reference: Prinz et al., *Markov models of molecular kinetics: Generation and validation*, J. Chem. Phys. **134**, 174105 (2011), [DOI](https://doi.org/10.1063/1.3565032), [full paper](https://publications.imp.fu-berlin.de/944/1/11_JCP_PrinzEtAl-MSM1.pdf).

## Instructor solution: more data AND more states
We retain the 160-point reference chain and collect independent equilibrium-started trajectories. Training and validation seeds never overlap. Starting at equilibrium makes symmetrized counts consistent without burn-in; the lesson's nonequilibrium population is used for propagation, not for estimating equilibrium counts.

We compare four-state and refined models at the same 100-step lag. All transition matrices below are fitted from trajectories. Exact projected matrices are used only to diagnose discretization error. The known equilibrium-conditional initializer is held fixed as in the assignment.

**Reproducibility:** eight training trajectories of two million MC steps each and eight separate validation trajectories. Prefix budgets share data and are not independent replicates. The chosen 80-state partition is a worked candidate, not a claim of an optimal partition.

In [ ]:
STEPS_PER_PATH = 2_000_000
training_seeds = list(range(1100,1108))
validation_seeds = list(range(2100,2108))
training = [simulate_mc(STEPS_PER_PATH, seed) for seed in training_seeds]
validation = [simulate_mc(STEPS_PER_PATH, seed) for seed in validation_seeds]
print("Training MC steps:",sum(len(p)-1 for p in training))
print("Validation MC steps:",sum(len(p)-1 for p in validation))
for name, paths in [("Training",training),("Validation",validation)]:
    occupancy=np.bincount(np.concatenate(paths),minlength=n)
    crossings=sum(np.count_nonzero(H4.argmax(1)[p[1:]]!=H4.argmax(1)[p[:-1]]) for p in paths)
    print(name,"minimum grid occupancy",occupancy.min(),"adjacent-step basin crossings",crossings)


### 1. Separate sampling error from discretization error
For each budget and partition, compute the maximum absolute error over all 16 A–D transition probabilities at 100, 200, …, 1000 MC steps. The exact projection score measures the error remaining even with unlimited sampling for that partition. The fitted score includes finite-data error; it can be accidentally smaller through cancellation.

In [ ]:
state_counts=[4,16,40,80]
budgets=[100_000,1_000_000,16_000_000]
exact_errors={M:exact_partition_score(M)[0] for M in state_counts}
results=[]
for budget in budgets:
    paths=[p[:budget//len(training)+1] for p in training]
    for M in state_counts:
        H=uniform_partition(M)
        try:
            fitted, stationary=fit_reversible_counts(transition_counts(paths,H,100))
            result=evaluate_fitted_msm(fitted,H,validation,100)
            err=result["max_error_vs_exact"]
        except ValueError as exc:
            print(budget,M,str(exc));err=np.nan
        results.append((budget,M,err,exact_errors[M]))
print("Training steps | States | Fitted max error | Exact partition error")
for row in results:print(f"{row[0]:14,d} | {row[1]:6d} | {row[2]:16.5f} | {row[3]:.5f}")
fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
for M in state_counts:
    rows=[r for r in results if r[1]==M]
    line,=ax.plot([r[0] for r in rows],[r[2] for r in rows],"o-",label=f"{M} states: fitted")
    ax.axhline(exact_errors[M],color=line.get_color(),ls=":",alpha=.6)
ax.axhline(.01,color="black",ls="--",label="0.01 target")
ax.set(xscale="log",yscale="log",xlabel="Total training MC steps",ylabel="Maximum basin CK error",title="Dotted lines: exact partition errors")
ax.legend(ncol=2);plt.show()


### 2. Visualize fitted modes and calculate timescales
The right eigenvectors satisfy $T\psi_i=\lambda_i\psi_i$ and are normalized with the fitted stationary probabilities, $\sum_s\widehat\pi_s\psi_i(s)^2=1$. Match signs to the reference for display. A negative value is not a probability or an intrinsic direction of flow.

For every lag, count transitions and fit a new matrix. Then $t_i=-	au/\log\lambda_i$ for $0<\lambda_i<1$. A plateau supports lag stability; a constant timescale constructed by powering one matrix would not be a test.

In [ ]:
M_solution=80
H_solution=uniform_partition(M_solution)
T_solution,pi_solution=fit_reversible_counts(transition_counts(training,H_solution,100))
values_solution,modes_solution=reversible_modes(T_solution,pi_solution)
reference_modes=psi_ref[:,:4]
for i in range(4):
    if np.dot(w*(H_solution@modes_solution[:,i]),reference_modes[:,i])<0:
        modes_solution[:,i]*=-1
np.testing.assert_allclose(modes_solution.T@(pi_solution[:,None]*modes_solution),np.eye(M_solution),atol=1e-10)
fig,axes=plt.subplots(5,1,figsize=(9,10),sharex=True,layout="constrained")
axes[0].plot(x,potential(x),"k");axes[0].set_ylabel("Potential")
for edge in np.linspace(-1,1,M_solution+1):axes[0].axvline(edge,color="gray",alpha=.15,lw=.5)
for i in range(4):
    axes[i+1].plot(x,reference_modes[:,i],"k--",label="Reference")
    axes[i+1].step(x,H_solution@modes_solution[:,i],where="mid",label="Count-fitted MSM")
    axes[i+1].set_ylabel(f"Mode {i+1}")
axes[1].set_ylim(.9,1.1)
axes[1].legend();axes[-1].set_xlabel("Position x");plt.show()
fitted_times=[]
for lag in lag_grid:
    fitted,stationary=fit_reversible_counts(transition_counts(training,H_solution,int(lag)))
    values,_=reversible_modes(fitted,stationary)
    fitted_times.append([-lag/np.log(v) if 0<v<1 else np.nan for v in values[1:4]])
fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
for i in range(3):
    line,=ax.plot(lag_grid,np.array(fitted_times)[:,i],"o-",label=f"Mode {i+2}")
    ax.axhline(-1/np.log(ell[i+1]),color=line.get_color(),ls="--")
ax.set(xlabel="Lag (MC steps)",ylabel="Implied timescale (MC steps)",title="80-state count fits; dashed = reference")
ax.legend();plt.show()
print("Lag-100 eigenvalues:",values_solution[:4])
print("Lag-100 timescales:",-100/np.log(values_solution[1:4]))


### 3. Validate every basin-to-basin transition
Retain the fine states during propagation: $B(k	au)=R\widehat T_M(	au)^kG$. Black points in the next grid are direct held-out trajectory estimates, not fitted predictions. The exact reference score is reported separately. Probability rows must sum to one.

In [ ]:
solution=evaluate_fitted_msm(T_solution,H_solution,validation,100)
np.testing.assert_allclose(solution["predicted"].sum(2),1,atol=1e-12)
assert solution["predicted"].min()>-1e-12
plot_ck_grid(solution["validation"],solution["predicted"],100,title="80 states: direct held-out counts vs MSM")
print("Maximum fitted error vs exact:",solution["max_error_vs_exact"])
print("Maximum fitted error vs held-out:",solution["max_error_vs_validation"])
print("Exact 80-state partition error:",exact_errors[80])
print("Meets 0.01 exact-reference target:",solution["max_error_vs_exact"]<.01)
# Independent training halves assess reproducibility with a smaller data budget.
for half in [training[:4],training[4:]]:
    fitted,_=fit_reversible_counts(transition_counts(half,H_solution,100))
    print("Independent 8-million-step fit, error vs exact:",evaluate_fitted_msm(fitted,H_solution,validation)["max_error_vs_exact"])


### 4. Uncertainty from whole trajectories
Bootstrap independent trajectories, not overlapping transition pairs. Cache each path's count matrix, resample eight paths with replacement, refit, and propagate. Separately resample the validation trajectories. The shaded bands below are pointwise 95% bootstrap intervals for diagonal probabilities. Invalid bootstrap fits are counted and reported; fitted intervals condition on valid fits and may understate uncertainty. Eight independent trajectories and 100 bootstrap resamples provide an illustrative uncertainty estimate, not guaranteed simultaneous coverage over all panels or precise tail probabilities.

In [ ]:
rng_boot=np.random.default_rng(3100)
train_counts=np.array([transition_counts([p],H_solution,100) for p in training])
validation_counts=np.array([[transition_counts([p],H4,int(k*100)) for k in K[1:]] for p in validation])
R,G=basin_maps(H_solution)
boot_predictions=[];boot_validation=[]
for _ in range(100):
    ids=rng_boot.integers(0,len(training),len(training))
    try:
        fitted,_=fit_reversible_counts(train_counts[ids].sum(0))
        boot_predictions.append([R@np.linalg.matrix_power(fitted,int(k))@G for k in K])
    except ValueError:
        boot_predictions.append(np.full((len(K),4,4),np.nan))
    ids=rng_boot.integers(0,len(validation),len(validation))
    counts=validation_counts[ids].sum(0)
    boot_validation.append(np.concatenate([np.eye(4)[None],counts/counts.sum(2,keepdims=True)],axis=0))
failed=sum(np.isnan(b).any() for b in boot_predictions)
print("Invalid bootstrap fits (missing states/connectivity):",failed,"of 100")
print("Fitted bands are conditional on valid fits; missing-state failures signal insufficient tail sampling.")
pred_limits=np.nanpercentile(boot_predictions,[2.5,97.5],axis=0)
val_limits=np.percentile(boot_validation,[2.5,97.5],axis=0)
fig,axes=plt.subplots(1,4,figsize=(14,3.6),layout="constrained")
for s,ax in enumerate(axes):
    ax.fill_between(K,pred_limits[0,:,s,s],pred_limits[1,:,s,s],color="#0072b2",alpha=.2)
    ax.fill_between(K,val_limits[0,:,s,s],val_limits[1,:,s,s],color="gray",alpha=.25)
    ax.plot(K,solution["predicted"][:,s,s],color="#0072b2",label="Fitted")
    ax.plot(K,solution["validation"][:,s,s],"k.",label="Held-out")
    ax.set(title=f"Basin {BASINS[s]}",xlabel="Lag steps k",ylim=(0,1.03))
axes[0].set_ylabel("Return probability");axes[0].legend();plt.show()
print("Largest validation interval width:",np.max(val_limits[1]-val_limits[0]))


### Interpretation and connection to the lecture
- Extending the four-state simulation estimates its short-lag transition probabilities more accurately, but leaves the memory introduced by merging many positions into a basin. Its exact CK discrepancy remains about 0.15.
- Refinement preserves information about position within each basin. The exact benchmark diagnoses how much discretization error remains; the count-fitted curve shows how much the available sampling changes the result.
- The slow mode separates A/B from C/D. The other leading modes primarily resolve exchange within those sides. Compare shapes and timescales rather than arbitrary eigenvector signs.
- The reported target result applies to these basin observables and times. Inspect the printed independent-fit errors and uncertainty intervals; if a run fails the target, increase sampling and reassess rather than changing the seed until it passes.
- The lecture amplitude calculation above uses the initial nonequilibrium distribution, not transition counts: $a_i=\sum_s p_{0,s}\psi_i(s)$. Its coefficients reconstruct the initial relative-density imbalance and evolve as $a_i\lambda_i^k$. This distinguishes learning the operator from using it to propagate an ensemble.

The four-state amplitudes and eigenvectors reproduced earlier belong to the four-state operator. Do not reuse those numerical coefficients as though they were the eigenbasis of the refined 80-state model.